# 4.7 章节练习参考答案

[客观题参考答案与解析](./04.07_answer.txt)

## 实验题参考代码

### 实验 19 代码示例：模型编译缓存

首次运行会在 `.torchair_cache_practice` 中生成 prompt 和 decode 缓存。请重启 Notebook 内核后再次运行该单元，对比两次端到端耗时，并结合日志与缓存目录确认是否命中缓存。

In [ ]:
import time
from pathlib import Path
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)
CACHE_DIR = ".torchair_cache_practice"


# 为 prompt/decode 两个固定 Shape 场景分别创建编译缓存。
class CachedPracticeModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(64, 64)
        self.cached_prompt = torch.npu.npugraph_ex.inference.cache_compile(
            self.prompt, cache_dir=CACHE_DIR
        )
        self.cached_decode = torch.npu.npugraph_ex.inference.cache_compile(
            self.decode, cache_dir=CACHE_DIR
        )

    def prompt(self, x):
        return torch.relu(self.linear(x))

    def decode(self, x):
        return torch.sigmoid(self.linear(x))


# 模型权重与输入统一为 float16，并切换到推理模式。
model = CachedPracticeModel().half().npu().eval()
prompt_x = torch.randn(8, 64, dtype=torch.float16).npu()
decode_x = torch.randn(1, 64, dtype=torch.float16).npu()

# 首次运行和重启内核后的缓存命中运行应分别记录耗时。
start = time.perf_counter()
with torch.no_grad():
    prompt_out = model.cached_prompt(prompt_x)
    decode_out = model.cached_decode(decode_x)
torch.npu.synchronize()
elapsed_ms = (time.perf_counter() - start) * 1000

print("prompt output:", tuple(prompt_out.shape))
print("decode output:", tuple(decode_out.shape))
print(f"本次端到端耗时: {elapsed_ms:.2f} ms")
print("缓存目录:", Path(CACHE_DIR).resolve())
print("提示：重启内核后再次运行本单元，记录缓存命中证据和耗时。")


### 实验 20 代码示例：单流与多流对比

下面的示例构造两条无数据依赖的矩阵计算分支。多流版本使用 Event 建立起止同步，并使用 `record_stream` 延长输入生命周期；`limit_core_num` 为每条 Stream 设置最大用核数。

In [ ]:
import time
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"


def single_stream(x1, w1, x2, w2):
    return torch.mm(x1, w1), torch.mm(x2, w2)


# 两个 Stream 分别执行无数据依赖的矩阵乘分支。
stream1 = torch.npu.Stream()
stream2 = torch.npu.Stream()
start_event = torch.npu.Event()
done1 = torch.npu.Event()
done2 = torch.npu.Event()


# 使用 Event 让非默认流等待输入准备完成，并让默认流等待分支结束。
def multi_stream(x1, w1, x2, w2):
    start_event.record()
    with torch.npu.stream(stream1):
        start_event.wait(stream1)
        with torch.npu.npugraph_ex.scope.limit_core_num(4, 8, stream1):
            out1 = torch.mm(x1, w1)
        x1.record_stream(stream1)
        w1.record_stream(stream1)
        done1.record()

    with torch.npu.stream(stream2):
        start_event.wait(stream2)
        with torch.npu.npugraph_ex.scope.limit_core_num(4, 8, stream2):
            out2 = torch.mm(x2, w2)
        x2.record_stream(stream2)
        w2.record_stream(stream2)
        done2.record()

    torch.npu.current_stream().wait_event(done1)
    torch.npu.current_stream().wait_event(done2)
    return out1, out2


# 预热后同步计时，避免把首次编译和异步下发误计为稳定性能。
def benchmark(fn, args, warmup=3, iters=20):
    for _ in range(warmup):
        fn(*args)
    torch.npu.synchronize()
    start = time.perf_counter()
    for _ in range(iters):
        fn(*args)
    torch.npu.synchronize()
    return (time.perf_counter() - start) * 1000 / iters


shape = (2048, 2048)
args = tuple(torch.randn(*shape, dtype=torch.float16).npu() for _ in range(4))
reference = single_stream(*args)
result = multi_stream(*args)
torch.npu.synchronize()
for got, expected in zip(result, reference):
    torch.testing.assert_close(got, expected, rtol=1e-3, atol=1e-3)

single_ms = benchmark(single_stream, args)
multi_ms = benchmark(multi_stream, args)
print(f"单流耗时: {single_ms:.3f} ms")
print(f"多流耗时: {multi_ms:.3f} ms")
print(f"加速比: {single_ms / multi_ms:.2f}x")
print("请继续使用 Profiler 时间线确认两条计算分支是否真正重叠。")
